In [25]:
import torch
import numpy as np
import pandas as pd

import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader,TensorDataset #do koncowego przygotowania danych
import pysam
import yfinance as yf #do danych finansowych
import os
from sklearn.preprocessing import MinMaxScaler #do skalowania

from sklearn.metrics import classification_report #do robienia raportów o uczeniu

import matplotlib.pyplot as plt

import seaborn as sns
import statsmodels.api as sm

import optuna
from optuna.trial import TrialState

import copy
import json 

import sys
sys.path.append('..')
import funkcje

device = torch.device("mps") if torch.backends.mps.is_available() else torch.device("cpu")

In [26]:
AUTOENCODER_PATH = '../models/best_autoencoder.pth'
TRADER_CONFIG_FILE = '../models/best_model_config.json'  
TRADER_MODEL_FILE = '../models/trader_best.pt'

SYMBOL = "BTC-USD"

AE_PARAMS = {
    'input_size': 6,    
    'hidden_size': 128,  
    'num_layers': 1,
    'seq_len': 16,     
    'latent_dim': 24    
}


In [27]:


class autocoder(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers, seq_len, latent_dim):
        super().__init__()
        self.seq_len = seq_len
 
        self.encoder_lstm = nn.LSTM(
            input_size=input_size, 
            hidden_size=hidden_size, 
            num_layers=num_layers, 
            batch_first=True
        )
        self.hidden_to_latent = nn.Linear(hidden_size, latent_dim)
        self.relu = nn.ReLU()
        self.latent_to_hidden = nn.Linear(latent_dim, hidden_size)
        self.decoder_lstm = nn.LSTM(
            input_size=hidden_size, 
            hidden_size=hidden_size, 
            num_layers=num_layers, 
            batch_first=True
        )
        self.output_layer = nn.Linear(hidden_size, input_size)

    def get_latent(self, x):
       
        _, (hidden_n, _) = self.encoder_lstm(x)
        last_hidden_state = hidden_n[-1]
        latent_vector = self.relu(self.hidden_to_latent(last_hidden_state))
        return latent_vector


class Trader(nn.Module):
    def __init__(self, input_dim, dim_1, dim_2, dropout):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(input_dim, dim_1),
            nn.BatchNorm1d(dim_1),      
            nn.ReLU(),
            nn.Dropout(dropout),          
            nn.Linear(dim_1, dim_2),
            nn.BatchNorm1d(dim_2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(dim_2, 1),        
            nn.Sigmoid()              
        )
        
    def forward(self, x):
        return self.network(x)


def load_system():


    if not os.path.exists(AUTOENCODER_PATH):
        raise FileNotFoundError(f"Brak pliku: {AUTOENCODER_PATH}")

    ae = autocoder(**AE_PARAMS).to(device)
    

    state_dict = torch.load(AUTOENCODER_PATH, map_location=device)
    ae.load_state_dict(state_dict)
    ae.eval()



    with open(TRADER_CONFIG_FILE, 'r') as f:
        config = json.load(f)


    if config['input_dim'] != AE_PARAMS['latent_dim']:
        print(f" OSTRZEŻENIE: Trader oczekuje {config['input_dim']} wejść, a Autoencoder daje {AE_PARAMS['latent_dim']}!")
    


    trader = Trader(
        input_dim=config['input_dim'],
        dim_1=config['dim_1'],
        dim_2=config['dim_2'],
        dropout=config['dropout']
    ).to(device)
    
    trader.load_state_dict(torch.load(TRADER_MODEL_FILE, map_location=device))
    trader.eval()
    
    return ae, trader


def get_market_data():


    data = yf.download(SYMBOL, period="1y", interval="1d", auto_adjust=True, progress=False)
    

    processed_df = funkcje.funkcja_do_danych(data)
    

    scaler = MinMaxScaler(feature_range=(-1, 1))
    scaled_data = scaler.fit_transform(processed_df.values)
    
    
    seq_len = AE_PARAMS['seq_len']
    if len(scaled_data) < seq_len:
        raise ValueError("Za mało danych do utworzenia sekwencji!")
        
    last_sequence = scaled_data[-seq_len:] 
    input_tensor = torch.FloatTensor(last_sequence).unsqueeze(0).to(device)
    
    last_timestamp = data.index[-1]
    last_date = last_timestamp.strftime('%Y-%m-%d')
    return input_tensor, last_date




In [28]:
ae_model, trader_model = load_system()
        

input_seq, date = get_market_data()
        
print(f"\n Analiza na dzień: {date}")

with torch.no_grad():
    latent_vector = ae_model.get_latent(input_seq)
    
with torch.no_grad():
    prediction = trader_model(latent_vector).item()
            

prob_percent = prediction * 100
decision = "CZEKAJ"
color = "\033[93m" 
        
if prediction > 0.75: 
    decision = "KUPUJ (LONG)"
    color = "\033[92m" 
elif prediction < 0.25:
    decision = "SPRZEDAWAJ (SHORT)"
    color = "\033[91m" 
print(f"Prawdopodobieństwo wzrostu: {prob_percent:.2f}%")
print(f"Decyzja systemu: {color}\033[1m{decision}\033[0m")
        
confidence = abs(prediction - 0.5) * 2
print(f"Pewność modelu: {confidence*100:.1f}%")



 Analiza na dzień: 2026-01-04
Prawdopodobieństwo wzrostu: 64.13%
Decyzja systemu: CZEKAJ
Pewność modelu: 28.3%
